In [1]:
from pyspark.sql import SparkSession


team = "team26"
warehouse = f"project/hive/warehouse_{team}"

spark = SparkSession.builder \
    .appName("{} - spark ML".format(team)) \
    .master("yarn") \
    .config("hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883") \
    .config("spark.sql.warehouse.dir", warehouse) \
    .config("spark.sql.avro.compression.codec", "snappy") \
    .enableHiveSupport() \
    .getOrCreate()



spark.sql("SHOW DATABASES").show(100, truncate=False)
spark.sql("SHOW TABLES IN team26_projectdb").show(100, truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 16:13:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/10 16:13:11 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
26/05/10 16:13:11 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


+---------------------+
|namespace            |
+---------------------+
|default              |
|ml_stage3            |
|retake1              |
|root_db              |
|show                 |
|team00_projectdb     |
|team0_dbms           |
|team0_projectdb      |
|team0db              |
|team0db1             |
|team0db2             |
|team11_projectdb     |
|team12_db            |
|team12_hive_projectdb|
|team12_projectdb     |
|team13_projectdb     |
|team13_projectdb_hive|
|team14_projectdb     |
|team15_projectdb     |
|team16_projectdb     |
|team17_projectdb     |
|team18_projectdb     |
|team19_projectdb     |
|team1_projectdb      |
|team20_projectdb     |
|team21_projectdb     |
|team21_projectdb_v2  |
|team21_projectdb_v3  |
|team21_projectdb_v4  |
|team22_projectdb     |
|team23_projectdb     |
|team24_projectdb     |
|team25_projectdb     |
|team26_projectdb     |
|team27_projectdb     |
|team28_hive_db       |
|team28_projectdb     |
|team29_projectdb     |
|team2_projectdb

In [2]:
print("spark.master =", spark.sparkContext.master)
print("deployMode =", spark.conf.get("spark.submit.deployMode", "(none)"))
print("applicationId =", spark.sparkContext.applicationId)

spark.master = yarn
deployMode = client
applicationId = application_1777989965680_4546


In [3]:
df = spark.table("team26_projectdb.chess_moves")
df.printSchema()
df.groupBy("final_result_class").count().show()

root
 |-- id: integer (nullable = true)
 |-- game_uuid: string (nullable = true)
 |-- game_url: string (nullable = true)
 |-- game_year: integer (nullable = true)
 |-- game_month: integer (nullable = true)
 |-- game_end_timestamp: long (nullable = true)
 |-- game_end_datetime_utc: long (nullable = true)
 |-- rated: boolean (nullable = true)
 |-- rules: string (nullable = true)
 |-- time_class: string (nullable = true)
 |-- time_control_raw: string (nullable = true)
 |-- time_control_base_seconds: integer (nullable = true)
 |-- time_control_increment_seconds: integer (nullable = true)
 |-- eco_url: string (nullable = true)
 |-- eco_code: string (nullable = true)
 |-- opening_name: string (nullable = true)
 |-- white_username: string (nullable = true)
 |-- black_username: string (nullable = true)
 |-- white_rating: integer (nullable = true)
 |-- black_rating: integer (nullable = true)
 |-- rating_diff: integer (nullable = true)
 |-- white_accuracy: float (nullable = true)
 |-- black_accu

+------------------+------+
|final_result_class| count|
+------------------+------+
|         white_win|299117|
|         black_win|307065|
|              draw| 54603|
+------------------+------+



In [4]:
from pyspark.sql import functions as F

def check_nulls(df):
    null_stats = (
        df.select([
            F.count(F.when(F.col(c).isNull(), 1)).alias(c)
            for c in df.columns
        ])
        .toPandas()
        .T
        .reset_index()
    )
    
    null_stats.columns = ["column", "null_count"]
    null_stats["null_pct"] = (null_stats["null_count"] / df.count()) * 100
    
    null_stats = null_stats[null_stats["null_count"] > 0].sort_values("null_pct", ascending=False)
    
    return null_stats

check_nulls(df)

26/05/10 16:13:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,column,null_count,null_pct
36,promotion_piece,658229,99.613187
22,black_accuracy,496570,75.148498
21,white_accuracy,496570,75.148498
82,pawn_shield_diff_before,239232,36.204212
80,white_pawn_shield_score_before,117616,17.799436
81,black_pawn_shield_score_before,113610,17.193187
85,king_tropism_diff_before,69641,10.539131
84,black_king_tropism_before,38624,5.845169
83,white_king_tropism_before,35990,5.446552
57,avg_time_spent_per_move_so_far,23669,3.581952


In [5]:
df.filter(F.col("king_tropism_diff_before").isNull()) \
  .select("game_uuid", "time_control_raw","time_class","king_tropism_diff_before", "ply_index").show(truncate=False)

+------------------------------------+----------------+----------+------------------------+---------+
|game_uuid                           |time_control_raw|time_class|king_tropism_diff_before|ply_index|
+------------------------------------+----------------+----------+------------------------+---------+
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL                    |14       |
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL                    |13       |
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL                    |12       |
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL                    |11       |
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL                    |8        |
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL                    |7        |
|4404b252-030a-11f0-b659-2f149001000f|180             |blitz     |NULL            

In [6]:
# filling in "avg_time_spent_per_move_so_far"

df = df.withColumn(
    "avg_time_spent_missing",
    F.col("avg_time_spent_per_move_so_far").isNull().cast("int")
)

med = (df
       .where(F.col("avg_time_spent_per_move_so_far").isNotNull())
       .groupBy("time_class")
       .agg(F.expr("percentile_approx(avg_time_spent_per_move_so_far, 0.5)").alias("med_avg_time"))
      )

df = (df.join(med, on="time_class", how="left")
        .withColumn(
            "avg_time_spent_per_move_so_far",
            F.coalesce(F.col("avg_time_spent_per_move_so_far"), F.col("med_avg_time"))
        )
        .drop("med_avg_time")
     )

Let's inspect *time_control_raw* values when *time_control_base_seconds* is Null

In [7]:
# find "time_control_raw" distribution when time_control_base_seconds is Null

df.filter(F.col("time_control_base_seconds").isNull()) \
  .select("time_control_raw","time_class","rated") \
  .groupBy("time_control_raw","time_class").count().show(50, False)

+----------------+----------+-----+
|time_control_raw|time_class|count|
+----------------+----------+-----+
|1/604800        |daily     |433  |
|1/86400         |daily     |2    |
+----------------+----------+-----+



In [8]:
df.groupBy("time_class").count().show()

+----------+------+
|time_class| count|
+----------+------+
|     blitz|302795|
|    bullet|259819|
|     rapid| 97736|
|     daily|   435|
+----------+------+



> We see that values of format A/B (for instance **1/604800**) belong to daily group of *time_class* feature and their quantity is 435. Also notice number of rows with *time_class* = **daily** is 435. So we can conclude that *time_control_raw* values are following the format of A/B when *time_class* is **daily**.

> Remember that *time_control_base_seconds* and *time_control_increment_seconds* were parsed from *time_control_raw* feature. It's easy to guess that A is time increment (in seconds) and B is time limit (for example 604800 seconds are 7 days).

In [9]:
tc = F.trim(F.col("time_control_raw"))

daily_slash_vals = ["1/86400", "1/604800"]

df = df.withColumn(
    "time_control_base_seconds",
    F.when(
        F.col("time_control_base_seconds").isNull() & (tc == F.lit("1/86400")),
        F.lit(86400)
    ).when(
        F.col("time_control_base_seconds").isNull() & (tc == F.lit("1/604800")),
        F.lit(604800)
    ).otherwise(F.col("time_control_base_seconds"))
)

df = df.withColumn(
    "time_control_increment_seconds",
    F.when(
        F.col("time_control_increment_seconds").isNull() &
        (tc.isin(daily_slash_vals)),
        F.lit(1)
    ).otherwise(F.col("time_control_increment_seconds"))
)

df = df.withColumn(
    "clock_remaining_pct",
    F.when(
        F.col("clock_remaining_pct").isNull() &
        (tc.isin(daily_slash_vals)) &
        F.col("side_to_move_clock_before").isNotNull() &
        (F.col("time_control_base_seconds") > 0),
        F.col("side_to_move_clock_before").cast("double") /
        F.col("time_control_base_seconds").cast("double")
    ).when(
        F.col("clock_remaining_pct").isNull() &
        (tc.isin(daily_slash_vals)) &
        (F.col("ply_index").isin([1, 2])),
        F.lit(1.0)
    ).otherwise(F.col("clock_remaining_pct"))
)

Let's also inspect values in *side_to_move_clock_before* and *is_in_time_trouble_30s*.

In [11]:
df.filter(F.col("side_to_move_clock_before").isNull()) \
  .select("time_control_raw", "side_to_move_clock_before", "is_in_time_trouble_30s", "clock_remaining_pct", "time_class", "ply_index").show()

+----------------+-------------------------+----------------------+-------------------+----------+---------+
|time_control_raw|side_to_move_clock_before|is_in_time_trouble_30s|clock_remaining_pct|time_class|ply_index|
+----------------+-------------------------+----------------------+-------------------+----------+---------+
|         1/86400|                     NULL|                  NULL|                1.0|     daily|        2|
|         1/86400|                     NULL|                  NULL|                1.0|     daily|        1|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        2|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        1|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        2|
|        1/604800|                     NULL|                  NULL|                1.0|     daily|        1|
|        1/604800| 

From this example, it becomes clear that side_to_move_clock_before must be filled with values of *time_control_base_seconds* because *ply_index* values indicate that these rows represent first move of opponents in the game. Also obvious that *is_in_time_trouble_30s* must be False for all these rows.

In [12]:
daily_slash = ["1/86400", "1/604800"]
tc = F.trim(F.col("time_control_raw"))
mask = tc.isin(daily_slash)

base_d = F.col("time_control_base_seconds").cast("double")
side_d = F.col("side_to_move_clock_before").cast("double")

side_filled = F.when(mask,
                     F.coalesce(side_d, base_d)
                    ).otherwise(side_d)

df = df.withColumn("side_to_move_clock_before", side_filled)

df = df.withColumn(
    "is_in_time_trouble_30s",
    F.when(
        mask & F.col("is_in_time_trouble_30s").isNull(),
        (F.col("side_to_move_clock_before") < F.lit(30)).cast("boolean")
    ).otherwise(F.col("is_in_time_trouble_30s"))
)

print("NULL side_to_move_clock_before (daily slash):",
      df.filter(mask & F.col("side_to_move_clock_before").isNull()).count())

print("NULL is_in_time_trouble_30s (daily slash):",
      df.filter(mask & F.col("is_in_time_trouble_30s").isNull()).count())

NULL side_to_move_clock_before (daily slash): 0


NULL is_in_time_trouble_30s (daily slash): 0


In [14]:
features = [
    "game_uuid",
    "rated",
    "rules",
    "time_class",
    "time_control_base_seconds",
    "time_control_increment_seconds",
    "white_rating",
    "black_rating",
    "rating_diff",
    "ply_index",
    "fullmove_number",
    "side_to_move",
    "is_capture",
    "is_check",
    "is_checkmate",
    "is_castling",
    "is_promotion",
    "piece_moved",
    "halfmove_clock_before",
    "legal_moves_count_before",
    "material_white_before",
    "material_black_before",
    "material_diff_before",
    "side_to_move_clock_before",
    "clock_remaining_pct",
    "avg_time_spent_per_move_so_far",
    "is_in_time_trouble_30s",
    "white_doubled_pawns_before",
    "black_doubled_pawns_before",
    "doubled_pawns_diff_before",
    "isolated_pawns_diff_before",
    "passed_pawns_diff_before",
    "white_can_castle_kingside_before",
    "white_can_castle_queenside_before",
    "black_can_castle_kingside_before",
    "black_can_castle_queenside_before",
    "in_check_before",
    "is_opening_phase",
    "is_middlegame_phase",
    "is_endgame_phase",
    "total_piece_count_before",
    "non_pawn_material_white_before",
    "bishops_pair_white_before",
    "bishops_pair_black_before",
    "final_result_class"
]

df_subset = df.select(*features)

In [15]:
check_nulls(df_subset)

,column,null_count,null_pct


In [17]:
df_subset.limit(10).toPandas()

,game_uuid,rated,rules,time_class,time_control_base_seconds,time_control_increment_seconds,white_rating,black_rating,rating_diff,ply_index,...,black_can_castle_queenside_before,in_check_before,is_opening_phase,is_middlegame_phase,is_endgame_phase,total_piece_count_before,non_pawn_material_white_before,bishops_pair_white_before,bishops_pair_black_before,final_result_class
0,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,18,...,True,False,True,False,False,26,19,True,True,black_win
1,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,17,...,True,False,True,False,False,26,19,True,True,black_win
2,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,16,...,True,False,True,False,False,27,22,True,True,black_win
3,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,15,...,True,True,True,False,False,27,22,True,True,black_win
4,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,14,...,True,False,True,False,False,27,22,True,True,black_win
5,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,13,...,True,False,True,False,False,27,22,True,True,black_win
6,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,12,...,True,False,True,False,False,28,22,True,True,black_win
7,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,11,...,True,False,True,False,False,28,22,True,True,black_win
8,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,10,...,True,False,True,False,False,28,22,True,True,black_win
9,9ef0155a-cdc9-11f0-b1f3-5623b701000f,True,chess,blitz,180,0,1659,1657,2,8,...,True,False,True,False,False,30,31,True,True,black_win


In [18]:
from pyspark.sql.types import BooleanType

label_col = "final_result_class"

cat_cols = ["rules", "time_class", "side_to_move", "piece_moved"]

bool_cols = [f.name for f in df_subset.schema.fields if isinstance(f.dataType, BooleanType)]

num_cols = [c for c in df_subset.columns if c not in cat_cols + bool_cols + [label_col] + ["game_uuid"]]

print("cat_cols:", cat_cols, '\n')
print("bool_cols:", bool_cols, '\n')
print("num_cols count:", num_cols)

cat_cols: ['rules', 'time_class', 'side_to_move', 'piece_moved'] 

bool_cols: ['rated', 'is_capture', 'is_check', 'is_checkmate', 'is_castling', 'is_promotion', 'is_in_time_trouble_30s', 'white_can_castle_kingside_before', 'white_can_castle_queenside_before', 'black_can_castle_kingside_before', 'black_can_castle_queenside_before', 'in_check_before', 'is_opening_phase', 'is_middlegame_phase', 'is_endgame_phase', 'bishops_pair_white_before', 'bishops_pair_black_before'] 

num_cols count: ['time_control_base_seconds', 'time_control_increment_seconds', 'white_rating', 'black_rating', 'rating_diff', 'ply_index', 'fullmove_number', 'halfmove_clock_before', 'legal_moves_count_before', 'material_white_before', 'material_black_before', 'material_diff_before', 'side_to_move_clock_before', 'clock_remaining_pct', 'avg_time_spent_per_move_so_far', 'white_doubled_pawns_before', 'black_doubled_pawns_before', 'doubled_pawns_diff_before', 'isolated_pawns_diff_before', 'passed_pawns_diff_before', 'total

In [19]:
# Type conversion (booleans and numeric)
for c in bool_cols:
    df_subset = df_subset.withColumn(c, F.col(c).cast("int").cast("double"))

for c in num_cols:
    df_subset = df_subset.withColumn(c, F.col(c).cast("double"))

In [20]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler

numeric_for_imputer = num_cols + bool_cols

imputed_cols = [c + "_imp" for c in numeric_for_imputer]

label_indexer = StringIndexer(
    inputCol=label_col, outputCol="label", handleInvalid="skip"
)

indexers = []
ohe = []
for c in cat_cols:
    idx_col = c + "_idx"
    ohe_col = c + "_ohe"
    indexers.append(
        StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid="keep")
    )
    ohe.append(
        OneHotEncoder(inputCol=idx_col, outputCol=ohe_col)
    )

imputer = Imputer(
    strategy="median",
    inputCols=numeric_for_imputer,
    outputCols=imputed_cols
)

numeric_assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="numeric_features"
)

ohe_cols = [c + "_ohe" for c in cat_cols]

final_assembler = VectorAssembler(
    inputCols=["numeric_features"] + ohe_cols,
    outputCol="features"
)

preprocess = [label_indexer] + indexers + ohe + [imputer, numeric_assembler, final_assembler]

In [21]:
SEED = 42

games = df_subset.select("game_uuid").distinct().withColumn("r", F.rand(SEED))
train_games = games.filter(F.col("r") < 0.7).select("game_uuid")
test_games  = games.filter(F.col("r") >= 0.7).select("game_uuid")

train_df = df_subset.join(train_games, "game_uuid", "inner").drop("game_uuid")
test_df  = df_subset.join(test_games,  "game_uuid", "inner").drop("game_uuid")

In [22]:
print("Train rows:", train_df.count(), "Test rows:", test_df.count())

Train rows: 468682 Test rows: 189332


# Now it is time for Modeling!

* ### Model 1 - Random Forest

In [24]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel


preprocess_pipeline = Pipeline(stages=preprocess)
preprocess_model = preprocess_pipeline.fit(train_df)

train_prepared = preprocess_model.transform(train_df).select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)
test_prepared = preprocess_model.transform(test_df).select("features", "label").persist(StorageLevel.MEMORY_AND_DISK)

train_prepared.count()
test_prepared.count()

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42,
    featureSubsetStrategy="sqrt",
    subsamplingRate=0.8,
    maxBins=64
)

paramGrid_rf = (ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 40, 60])             # 3
    .addGrid(rf.maxDepth, [3, 6, 9])              # 3
    .addGrid(rf.minInstancesPerNode, [5, 10, 20]) # 3  => 27
    .build()
)

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)


cv_rf = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator_acc,
    numFolds=3,
    seed=42,
    parallelism=1
)

cvModelRf = cv_rf.fit(train_prepared)

best_rf_model = cvModelRf.bestModel
pred_rf = best_rf_model.transform(test_prepared)
acc_rf = evaluator_acc.evaluate(pred_rf)
f1_rf = evaluator_f1.evaluate(pred_rf)


print("RF -> TEST acc:", acc_rf, "TEST f1:", f1_rf)
print("Params:", best_rf_model.extractParamMap())

26/05/10 16:19:56 WARN DAGScheduler: Broadcasting large task binary with size 1015.2 KiB
26/05/10 16:20:11 WARN DAGScheduler: Broadcasting large task binary with size 1665.6 KiB
26/05/10 16:20:33 WARN DAGScheduler: Broadcasting large task binary with size 1173.1 KiB
26/05/10 16:21:23 WARN DAGScheduler: Broadcasting large task binary with size 1010.7 KiB
26/05/10 16:21:38 WARN DAGScheduler: Broadcasting large task binary with size 1648.5 KiB
26/05/10 16:21:58 WARN DAGScheduler: Broadcasting large task binary with size 1111.8 KiB
26/05/10 16:22:48 WARN DAGScheduler: Broadcasting large task binary with size 1001.8 KiB
26/05/10 16:23:03 WARN DAGScheduler: Broadcasting large task binary with size 1611.6 KiB
26/05/10 16:23:24 WARN DAGScheduler: Broadcasting large task binary with size 1004.5 KiB
26/05/10 16:28:10 WARN DAGScheduler: Broadcasting large task binary with size 1044.9 KiB
26/05/10 16:28:28 WARN DAGScheduler: Broadcasting large task binary with size 1791.1 KiB
26/05/10 16:28:54 WAR

TypeError: 'NoneType' object is not iterable

* ### Model 2 - Naive Bayes (Multinomial) 

In [29]:
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.feature import Binarizer, VectorAssembler


binarizer = Binarizer(
    inputCol="features",
    outputCol="features_bin",
    threshold=0.0
)

nb = NaiveBayes(
    featuresCol="features_bin",
    labelCol="label",
    predictionCol="prediction",
    modelType="bernoulli"
)

nb_pipeline = Pipeline(stages=[binarizer, nb])


paramGrid_nb = (ParamGridBuilder()
    .addGrid(nb.smoothing, [0.5, 1.0, 2.0])    # 3
    .addGrid(binarizer.threshold, [0.0, 0.5, 1.0])    # 3
    .addGrid(nb.modelType, ["bernoulli", "multinomial", "gaussian"])    # 3 => 27
    .build())


cv_nb = CrossValidator(
    estimator=nb_pipeline,
    estimatorParamMaps=paramGrid_nb,
    evaluator=evaluator_acc,
    numFolds=3,
    parallelism=1,
    seed=SEED
)

cvModelNb = cv_nb.fit(train_prepared)
best_nb_model = cvModelNb.bestModel

pred_nb = best_nb_model.transform(test_prepared)
acc_nb = evaluator_acc.evaluate(pred_nb)
f1_nb = evaluator_f1.evaluate(pred_nb)


print("Naive Bayes -> TEST acc:", acc_nb, "TEST f1:", f1_nb)
print("Params:", best_nb_model.extractParamMap())

Naive Bayes – TEST Accuracy: 0.6709557576611409  TEST F1: 0.6709557576611409
Best model parameters:
{Param(parent='NaiveBayes_6b7927129764', name='featuresCol', doc='features column name.'): 'features_bin', Param(parent='NaiveBayes_6b7927129764', name='labelCol', doc='label column name.'): 'label', Param(parent='NaiveBayes_6b7927129764', name='modelType', doc='The model type which is a string (case-sensitive). Supported options: multinomial (default), bernoulli and gaussian.'): 'bernoulli', Param(parent='NaiveBayes_6b7927129764', name='predictionCol', doc='prediction column name.'): 'prediction', Param(parent='NaiveBayes_6b7927129764', name='probabilityCol', doc='Column name for predicted class conditional probabilities. Note: Not all models output well-calibrated probability estimates! These probabilities should be treated as confidences, not precise probabilities.'): 'probability', Param(parent='NaiveBayes_6b7927129764', name='rawPredictionCol', doc='raw prediction (a.k.a. confidence

* ### Model 3 - Logistic Regression

In [50]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StandardScaler

spark.conf.set("spark.ml.crossValidator.parallelism", "1")


scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=False, withStd=True)
scaler_model = scaler.fit(train_prepared)
train_scaled = scaler_model.transform(train_prepared).select("features_scaled", "label") \
                     .persist(StorageLevel.MEMORY_AND_DISK)
test_scaled = scaler_model.transform(test_prepared).select("features_scaled", "label") \
                    .persist(StorageLevel.MEMORY_AND_DISK)
train_scaled.count()
test_scaled.count()

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="label",
    maxIter=50,            
    family="multinomial",     
    standardization=False
)

paramGrid_lr = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1, 1.0])          # 3
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])    # 3
    .addGrid(lr.tol, [1e-4, 1e-3, 1e-2])            # 3 => 27
    .build())


cv_lr = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=evaluator_acc,
    numFolds=3,
    parallelism=1,
    seed=SEED
)


cvModelLr = cv_lr.fit(train_scaled)
best_lr_model = cvModelLr.bestModel

pred_lr = best_lr_model.transform(test_scaled)
acc_lr = evaluator_acc.evaluate(pred_lr)
f1_lr = evaluator_f1.evaluate(pred_lr)



print("Logistic Regression -> TEST acc:", acc_lr, "TEST f1:", f1_lr)
print("Params:", best_lr_model.extractParamMap())

train_prepared.unpersist()
test_prepared.unpersist()
train_scaled.unpersist()
test_scaled.unpersist()

Logistic Regression - TEST Accuracy: 0.6806
Logistic Regression - TEST F1: 0.6540
Best parameters:
  regParam = 0.01
  elasticNetParam = 0.0
  maxIter = 50


DataFrame[features: vector, label: double]